# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding chosen #1 — "What Predicts Growth?" (ML Appendix, p.29).**
The paper reports a logistic regression at **71% holdout accuracy** separating growing from
declining pages.

**My methodology question:** what's the base rate this is being compared against? Finding #1
(p.6) reports 74.8K growing pages vs. 45.6K declining ones -- a base rate of roughly **62%** for
the majority class ("up"). If a model that always guesses "growing" would already score ~62%,
then 71% accuracy is genuinely real skill, but it's only about **9 points of lift over the naive
guess**, not 71 points of skill. The paper states the accuracy number but doesn't state the base
rate next to it in that section, so a reader has to hunt two pages back to contextualize it. This
is asked in the same spirit the `hunting-leakage-and-validating` skill teaches: "always print the
base rate next to it" -- not a criticism of the finding itself, just a request to make the
comparison easier for the reader to see without cross-referencing.

**Finding chosen #2 — "What Predicts Health?" (ML Appendix, p.27).**
A Random Forest ranks **Average Position (43%)**, **Impressions (32%)**, **Scroll Depth (15%)**,
and **CTR (8%)** as the top features for predicting Health Score -- together, 98% of the
model's importance.

**My methodology question:** the paper's own Health Score formula (stated on p.5) is *literally*
`Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts)`. Three of the four
top "predictive" features -- Position, Impressions, CTR -- are direct arithmetic inputs to the
label itself, and the fourth (Scroll Depth) is the label's remaining component. So is this model
predicting an independent outcome, or mostly re-deriving the coefficients of its own label's
formula? The paper does flag this honestly in one line ("importance is descriptive rather than
causal") -- my question is just whether that caveat should be stronger, given that ~98% of the
model's importance sits on features that are definitionally part of the target, which is closer
to the `hunting-leakage-and-validating` skill's "label-derived features" pattern than a genuine
external-signal finding.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [3]:
import os, getpass
import numpy as np
import pandas as pd
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
RANDOM_STATE = 42

# Same contract as w03/w04/w05
feature_frame = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                            AS imp_h1,
        SUM(gsc_clicks)                                                 AS clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)               AS ctr_h1,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0)       AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days_h1
    FROM {fact_march}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

label_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31') AS imp_h2
    FROM {fact_march}
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(label_frame, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining_proxy"] = (data["imp_h2"] < 0.8 * data["imp_h1"]).astype(int)
data = data.dropna(subset=["ctr_h1", "avg_position_h1"]).reset_index(drop=True)

FEATURES = ["imp_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
X_all = data[FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0)
y_all = data["is_declining_proxy"].astype(int)
print(f"{len(data):,} rows | decline rate: {y_all.mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,475 rows | decline rate: 0.296


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

def fit_and_score(X_train, X_test, y_train, y_test, label):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ])
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        "split": label,
        "precision_at_50": precision_at_k(y_test, proba, 50),
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
    }

# BEFORE: naive random row-level split -- ignores that many rows share the same client
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all
)
before = fit_and_score(X_tr_rand, X_te_rand, y_tr_rand, y_te_rand, "BEFORE: random row split")

# AFTER: honest client-holdout split, same as w05_model.ipynb
def client_holdout_split(frame):
    all_indices = np.arange(len(frame))
    client_series = frame["client_hash_id"].astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(unique_clients)
    n_test = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:n_test])
    test_mask = client_series.isin(test_clients).to_numpy()
    return all_indices[~test_mask], all_indices[test_mask]

train_idx, test_idx = client_holdout_split(data)
after = fit_and_score(
    X_all.iloc[train_idx], X_all.iloc[test_idx], y_all.iloc[train_idx], y_all.iloc[test_idx],
    "AFTER: client-holdout split",
)

before_after = pd.DataFrame([before, after]).set_index("split").round(3)
before_after

,precision_at_50,roc_auc,avg_precision
split,,,
BEFORE: random row split,0.6,0.587,0.367
AFTER: client-holdout split,0.5,0.474,0.395


In [10]:
gap = before_after.loc["BEFORE: random row split", "precision_at_50"] - before_after.loc["AFTER: client-holdout split", "precision_at_50"]
print(f"Gap (random split - client-holdout split) on Precision@50: {gap:.3f}")
print("Gap (random split - client-holdout split) on Precision@50: 0.100")
print("The random split scores higher on Precision@50 (0.6 vs 0.5) and ROC-AUC (0.587 vs 0.474)")
print("-- that gap is memorization: with a random split, rows from the same client can appear in")
print("both train and test, so the model partly learns client-specific quirks rather than a")
print("pattern that generalizes to a brand-new client. Once client-holdout removes that leak,")
print("ROC-AUC actually drops below 0.5 -- barely better than random ranking on unseen clients.")
print("Average precision moved the other way (0.367 vs 0.395), which is likely just noise: the")
print("client-holdout test set has far fewer distinct clients, so a metric like average precision")
print("that's sensitive to exact ranking order is less stable on a smaller, noisier test set.")

Gap (random split - client-holdout split) on Precision@50: 0.100
Gap (random split - client-holdout split) on Precision@50: 0.100
The random split scores higher on Precision@50 (0.6 vs 0.5) and ROC-AUC (0.587 vs 0.474)
-- that gap is memorization: with a random split, rows from the same client can appear in
both train and test, so the model partly learns client-specific quirks rather than a
pattern that generalizes to a brand-new client. Once client-holdout removes that leak,
ROC-AUC actually drops below 0.5 -- barely better than random ranking on unseen clients.
Average precision moved the other way (0.367 vs 0.395), which is likely just noise: the
client-holdout test set has far fewer distinct clients, so a metric like average precision
that's sensitive to exact ranking order is less stable on a smaller, noisier test set.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [9]:
honest_features = FEATURES
leaky_features = FEATURES + ["imp_h2"]  # the label's own input, smuggled in as a trap

def quick_auc(cols):
    X = data[cols]
    y = data["is_declining_proxy"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_auc(honest_features)
leaky_auc = quick_auc(leaky_features)
print(f"honest ROC-AUC (final Week-5 feature set): {honest_auc:.3f}")
print(f"leaky ROC-AUC (+ imp_h2 smuggled in):       {leaky_auc:.3f}")
print(f"gap: {leaky_auc - honest_auc:.3f}")
print("The leaky version jumps all the way to a perfect 1.000 AUC once imp_h2 is smuggled in,")
print("vs. 0.584 for the honest feature set -- a 0.416 gap. A perfect score is the clearest")
print("possible confession: is_declining_proxy is DEFINED directly from imp_h2 (imp_h2 < 0.8 *")
print("imp_h1), so giving the model imp_h2 lets it reconstruct the label almost exactly rather")
print("than actually predicting anything. This confirms the test harness itself works correctly")
print("-- it should catch a leak this severe, and it did. imp_h2 was only ever used for this")
print("demonstration and is not part of the real feature set used anywhere else in this project.")
print("No FlyRank product-decision fields (health_score, priority_score, action_type,")
print("refresh_tier) made it into the final features either.")


# Product-flag check: confirm none of FlyRank's own decision fields ever entered the feature set.
product_flags = {"health_score", "priority_score", "action_type", "refresh_tier"}
print(f"\nProduct-decision fields in FEATURES: {product_flags & set(FEATURES)} (should be empty set)")

honest ROC-AUC (final Week-5 feature set): 0.584
leaky ROC-AUC (+ imp_h2 smuggled in):       1.000
gap: 0.416
The leaky version jumps all the way to a perfect 1.000 AUC once imp_h2 is smuggled in,
vs. 0.584 for the honest feature set -- a 0.416 gap. A perfect score is the clearest
possible confession: is_declining_proxy is DEFINED directly from imp_h2 (imp_h2 < 0.8 *
imp_h1), so giving the model imp_h2 lets it reconstruct the label almost exactly rather
than actually predicting anything. This confirms the test harness itself works correctly
-- it should catch a leak this severe, and it did. imp_h2 was only ever used for this
demonstration and is not part of the real feature set used anywhere else in this project.
No FlyRank product-decision fields (health_score, priority_score, action_type,
refresh_tier) made it into the final features either.

Product-decision fields in FEATURES: set() (should be empty set)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*





**Original (from `w04_signal_audit.ipynb`, Section 4):**
> "CTR is the one clean, trustworthy signal here — low-CTR pages decline at roughly 1.5x the rate
> of high-CTR ones, and it's monotonic across the whole range."

**Rewritten:**
> "In this one-month observation window, CTR was the most consistent measured signal: pages in
> the lowest CTR quartile showed roughly 1.5x the observed decline rate of the highest quartile,
> and the relationship was directional (monotonic) across all four quartiles. This is a
> single-window, decision-support signal, not a proven causal driver of decline — a page's CTR
> and its future trajectory could both be caused by some third factor (e.g. a recent SERP
> feature change) rather than CTR directly causing the decline."

**Why the rewrite matters:** "clean, trustworthy" is doing a lot of work in the original --
it reads as a settled fact rather than one month's observation. The rewrite keeps the actual
finding (the 1.5x gap, the monotonic pattern) but drops the implied permanence and causation,
and names the specific alternative explanation (confounding) instead of letting the strong
adjective imply causation was ruled out.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.